In [3]:
import torch
import json
import os
import requests
import numpy as np
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification, AutoModel
from huggingface_hub import hf_hub_download
import torch.nn as nn

# --- CONFIGURATION ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NULL_TOKEN = "[NULL]"
MAX_LEN = 128

# REPLACE THESE WITH YOUR HUGGING FACE REPO IDs
EXTRACTOR_REPO = "affan002/roberta-extractor-task2-eng" 
PAIRING_REPO   = "affan002/roberta-pairing-task2-eng"
REGRESSOR_REPO = "affan002/xlm-roberta-large-task1-zho-eng" 

# --- MODEL CLASS DEFINITION (Required for loading Regressor) ---
class TransformerVARegressor(nn.Module):
    def __init__(self, model_name, dropout=0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.reg_head = nn.Linear(self.backbone.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0]
        x = self.dropout(cls_output)
        return self.reg_head(x)

# --- DATASET TASKS ---
TASKS = [
    {"lang": "eng", "domain": "laptop", "url": "https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/subtask_3/eng/eng_laptop_dev_task3.jsonl"},
    {"lang": "eng", "domain": "restaurant", "url": "https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/subtask_3/eng/eng_restaurant_dev_task3.jsonl"},
    ]

## Loading extractor, pairer and VA model 

In [4]:
from huggingface_hub import hf_hub_download , login
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import json
import torch
# Login with your token for private models
login(token=os.environ["HF_TOKEN"])  # <--- Add your token here
print("✅ Logged in to Hugging Face")


# --- 1. Load Extractor ---
print("Loading Extractor...")
ext_tokenizer = AutoTokenizer.from_pretrained(EXTRACTOR_REPO)
ext_model = AutoModelForTokenClassification.from_pretrained(EXTRACTOR_REPO).to(DEVICE)
ext_model.eval()
id2label = ext_model.config.id2label

# --- 2. Load Pairer ---
print("Loading Pairer...")
pair_tokenizer = AutoTokenizer.from_pretrained(PAIRING_REPO)
pair_model = AutoModelForSequenceClassification.from_pretrained(PAIRING_REPO).to(DEVICE)
pair_model.eval()

# --- 3. Load Regressor (Custom Class) ---
print("Loading Regressor...")
# We initialize with base XLM-R structure first
reg_model = TransformerVARegressor(model_name="xlm-roberta-large") # Or 'base' if you used base
reg_tokenizer = AutoTokenizer.from_pretrained(REGRESSOR_REPO)

# Resize embeddings (Crucial!)
reg_model.backbone.resize_token_embeddings(len(reg_tokenizer))

# Download weights manually because it's a custom class
try:
    # Try safetensors first
    from safetensors.torch import load_file
    model_path = hf_hub_download(repo_id=REGRESSOR_REPO, filename="model.safetensors")
    state_dict = load_file(model_path)
except:
    # Fallback to pytorch_model.bin
    model_path = hf_hub_download(repo_id=REGRESSOR_REPO, filename="pytorch_model.bin")
    state_dict = torch.load(model_path, map_location=DEVICE)

reg_model.load_state_dict(state_dict)
reg_model.to(DEVICE)
reg_model.eval()

print("✅ All Models Loaded Successfully!")

✅ Logged in to Hugging Face
Loading Extractor...


tokenizer_config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/889 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


pytorch_model.bin:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

✅ All Models Loaded Successfully!


In [5]:
def run_extractor(text):
    # 1. Prepend Double NULLs (Crucial: Match Training!)
    aug_text = f"{NULL_TOKEN} {NULL_TOKEN} {text}"
    
    inputs = ext_tokenizer(
        aug_text, 
        return_tensors="pt", 
        truncation=True, 
        max_length=128
    ).to(DEVICE)
    
    with torch.no_grad():
        outputs = ext_model(**inputs)
    
    # Get Predictions
    preds = torch.argmax(outputs.logits, dim=2)[0].cpu().numpy()
    tokens = ext_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    
    # Helper to clean RoBERTa tokens
    def clean(toks): return ext_tokenizer.convert_tokens_to_string(toks).strip()

    aspects, opinions = [], []
    curr_word, curr_type = [], None
    
    for t, p in zip(tokens, preds):
        label = id2label[p]
        
        # Skip special tokens (CLS, SEP, PAD)
        if t in [ext_tokenizer.cls_token, ext_tokenizer.sep_token, ext_tokenizer.pad_token]:
            continue
        
        # --- EXPLICIT NULL CHECK ---
        # If the model tagged the special [NULL] token, we record it immediately.
        # This handles Implicit Aspects/Opinions.
        if t == NULL_TOKEN:
            if "ASP" in label: aspects.append("[NULL]")
            if "OPI" in label: opinions.append("[NULL]")
            continue # Don't add [NULL] to current_word list
        
        # Standard BIO Logic
        if label.startswith("B-"):
            if curr_word: 
                w = clean(curr_word)
                if curr_type == "ASP": aspects.append(w)
                elif curr_type == "OPI": opinions.append(w)
            curr_word = [t]
            curr_type = label.split("-")[1] # 'ASP' or 'OPI'
            
        elif label.startswith("I-") and curr_type == label.split("-")[1]:
            curr_word.append(t)
            
        else: # 'O' tag or mismatch
            if curr_word:
                w = clean(curr_word)
                if curr_type == "ASP": aspects.append(w)
                elif curr_type == "OPI": opinions.append(w)
            curr_word = []; curr_type = None
            
    # Catch the last word
    if curr_word:
        w = clean(curr_word)
        if curr_type == "ASP": aspects.append(w)
        elif curr_type == "OPI": opinions.append(w)
        
    return list(set(aspects)), list(set(opinions))

In [6]:
def run_pairer(text, aspect, opinion):
    
    # Input A: [NULL] [NULL] Text
    aug_text = f"{NULL_TOKEN} {NULL_TOKEN} {text}"
    
    # Input B: Aspect </s> Opinion
    pair_text = f"{aspect} {pair_tokenizer.sep_token} {opinion}"
    
    inputs = pair_tokenizer(
        aug_text, 
        pair_text, 
        return_tensors="pt", 
        truncation=True, 
        max_length=MAX_LEN
    ).to(DEVICE)
    
    with torch.no_grad():
        outputs = pair_model(**inputs)
    
    # Label 1 = Valid Pair
    probs = torch.softmax(outputs.logits, dim=1)[0]
    pred_label = torch.argmax(probs).item()
    
    # Debug Info (Optional: helps you see confidence)
    print(f"   Pair: {aspect} + {opinion} -> Score: {probs[1]:.4f}")

    return pred_label == 1

In [7]:
def run_regressor(text, aspect, domain):
    # Input: [DOMAIN] Aspect: Sentence
    # Match your training format exactly!
    input_text = f"[{domain.upper()}] {aspect}: {text}"
    
    inputs = reg_tokenizer(
        input_text, 
        return_tensors="pt", 
        truncation=True, 
        max_length=128
    ).to(DEVICE)
    
    with torch.no_grad():
        outputs = reg_model(inputs["input_ids"], inputs["attention_mask"])
        
    scores = outputs.cpu().numpy()[0]
    v = max(1.0, min(9.0, scores[0]))
    a = max(1.0, min(9.0, scores[1]))
    
    return f"{v:.2f}#{a:.2f}"

## loading the models for category

In [8]:
# ============================================
# CELL: Load Models from Hugging Face and Evaluate
# ============================================
from huggingface_hub import hf_hub_download , login
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import json
import torch

# Login with your token for private models
login(token=os.environ["HF_TOKEN"])  # <--- Add your token here
print("✅ Logged in to Hugging Face")

print("="*60)
print("LOADING MODELS FROM HUGGING FACE HUB")
print("="*60)

# Define your model repository IDs
checkpoint = "xlm-roberta-base"  # or whatever checkpoint you used
entity_repo_id = f"hassanshahzad2003/{checkpoint}_task3_entity_aug"
attr_repo_id = f"hassanshahzad2003/{checkpoint}_task3_attribute_aug"

# ===== Load Entity Model =====
print("\n--- Loading Entity Model ---")
entity_model_loaded = AutoModelForSequenceClassification.from_pretrained(entity_repo_id)
tokenizer_loaded = AutoTokenizer.from_pretrained(entity_repo_id)

# Download and load label mappings
entity_label_file = hf_hub_download(repo_id=entity_repo_id, filename="label_mappings.json")
with open(entity_label_file, 'r') as f:
    entity_labels = json.load(f)
    entity2id_loaded = entity_labels['entity2id']
    id2entity_loaded = {int(k): v for k, v in entity_labels['id2entity'].items()}

print(f"✅ Entity model loaded from: {entity_repo_id}")
print(f"   Entities: {list(entity2id_loaded.keys())}")

# ===== Load Attribute Model =====
print("\n--- Loading Attribute Model ---")
attribute_model_loaded = AutoModelForSequenceClassification.from_pretrained(attr_repo_id)

# Download and load label mappings
attr_label_file = hf_hub_download(repo_id=attr_repo_id, filename="label_mappings.json")
with open(attr_label_file, 'r') as f:
    attr_labels = json.load(f)
    attribute2id_loaded = attr_labels['attribute2id']
    id2attribute_loaded = {int(k): v for k, v in attr_labels['id2attribute'].items()}

print(f"✅ Attribute model loaded from: {attr_repo_id}")
print(f"   Attributes: {len(attribute2id_loaded)} classes")

# Move models to device
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
entity_model_loaded.to(device)
attribute_model_loaded.to(device)
print(f"\n✅ Models moved to: {device}")

# ===== Update Prediction Function to Use Loaded Models =====
def predict_category_loaded(text, aspect):
    """
    Predict the full category (Entity#Attribute) using loaded models from HF Hub.
    """
    # Step 1: Predict Entity
    entity_model_loaded.eval()
    inputs = tokenizer_loaded(text, aspect, truncation=True, max_length=128, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = entity_model_loaded(**inputs)
    entity_id = torch.argmax(outputs.logits, dim=-1).item()
    entity = id2entity_loaded[entity_id]

    # Step 2: Predict Attribute
    attribute_model_loaded.eval()
    combined_text = f"{text} [SEP] {aspect} [SEP] {entity}"
    inputs = tokenizer_loaded(combined_text, truncation=True, max_length=128, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = attribute_model_loaded(**inputs)
    attribute_id = torch.argmax(outputs.logits, dim=-1).item()
    attribute = id2attribute_loaded[attribute_id]

    category = f"{entity}#{attribute}"
    return entity, attribute, category

# ===== Test Single Prediction =====
print("\n--- Testing Loaded Models ---")
test_text = "this unit is pretty and stylish, so my high school daughter was attracted to it for that reason."
test_aspect = "unit"

entity, attribute, category = predict_category_loaded(test_text, test_aspect)
print(f"\nTest Prediction:")
print(f"Text: {test_text}")
print(f"Aspect: {test_aspect}")
print(f"Predicted Entity: {entity}")
print(f"Predicted Attribute: {attribute}")
print(f"Predicted Category: {category}")

# ===== Run Full Evaluation with Loaded Models =====
print("\n" + "="*60)
print("RUNNING EVALUATION WITH LOADED MODELS")
print("="*60)

# Update global variables to use loaded models for evaluation
entity_model = entity_model_loaded
attribute_model = attribute_model_loaded
tokenizer = tokenizer_loaded
entity2id = entity2id_loaded
id2entity = id2entity_loaded
attribute2id = attribute2id_loaded
id2attribute = id2attribute_loaded

# Update predict_category function to use loaded models
predict_category = predict_category_loaded



✅ Logged in to Hugging Face
LOADING MODELS FROM HUGGING FACE HUB

--- Loading Entity Model ---


config.json:   0%|          | 0.00/2.33k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

✅ Entity model loaded from: hassanshahzad2003/xlm-roberta-base_task3_entity_aug
   Entities: ['AMBIENCE', 'BATTERY', 'COMPANY', 'CPU', 'DISPLAY', 'DRINKS', 'FANS&COOLING', 'FANS_COOLING', 'FOOD', 'GRAPHICS', 'HARDWARE', 'HARD_DISC', 'HARD_DISK', 'KEYBOARD', 'LAPTOP', 'LOCATION', 'MEMORY', 'MOTHERBOARD', 'MOUSE', 'MULTIMEDIA_DEVICES', 'OPTICAL_DRIVES', 'OS', 'OUT_OF_SCOPE', 'Out_Of_Scope', 'PORTS', 'POWER_SUPPLY', 'RESTAURANT', 'SERVICE', 'SHIPPING', 'SOFTWARE', 'SUPPORT', 'WARRANTY', '地点', '服务', '氛围', '食物', '餐厅', '饮料']

--- Loading Attribute Model ---


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

label_mappings.json:   0%|          | 0.00/840 [00:00<?, ?B/s]

✅ Attribute model loaded from: hassanshahzad2003/xlm-roberta-base_task3_attribute_aug
   Attributes: 16 classes

✅ Models moved to: cuda

--- Testing Loaded Models ---

Test Prediction:
Text: this unit is pretty and stylish, so my high school daughter was attracted to it for that reason.
Aspect: unit
Predicted Entity: LAPTOP
Predicted Attribute: DESIGN_FEATURES
Predicted Category: LAPTOP#DESIGN_FEATURES

RUNNING EVALUATION WITH LOADED MODELS


## Pipeline

In [9]:
def fetch_data(url):
    try:
        response = requests.get(url)
        response.raise_for_status()
        return [json.loads(line) for line in response.text.strip().split('\n') if line]
    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return []

In [10]:
output_dir = "subtask_3"
os.makedirs(output_dir, exist_ok=True)

print(f"Starting Inference Pipeline...")

for task in TASKS:
    lang = task['lang']
    domain = task['domain']
    filename = f"pred_{lang}_{domain}.jsonl"
    print(f"\nProcessing {lang}-{domain}...")
    
    # 1. Load Data from GitHub
    data = fetch_data(task['url'])
    results = []
    
    for entry in tqdm(data):
        text = entry['Text']
        
        # --- STEP 1: EXTRACT ---
        # Get lists of candidates
        aspects, opinions = run_extractor(text)
        
        triplets = []
        
        # --- STEP 2: PAIR LOOP (Cartesian Product) ---
        for asp in aspects:
            for opi in opinions:
                
                # # Skip if both are NULL (usually noise)
                # if asp == "[NULL]" and opi == "[NULL]": continue
                
                # --- STEP 3: VALIDATE ---
                # Ask Model 2: "Is this pair valid?"
                if run_pairer(text, asp, opi):
                    
                    # --- STEP 4: SCORE ---
                    # Ask Model 3: "What is the VA?"
                    va_score = run_regressor(text, asp, domain)

                    entity, attribute, category = predict_category_loaded(text, asp)
                    
                    # Clean format for output
                    final_asp = "NULL" if asp == "[NULL]" else asp
                    final_opi = "NULL" if opi == "[NULL]" else opi
                    
                    
                    
                    triplets.append({
                        "Aspect": final_asp,
                        "Category": category,
                        "Opinion": final_opi,
                        "VA": va_score
                    })
        
        results.append({
            "ID": entry['ID'],
            "Quadruplet": triplets
        })
        
    # Save to File
    with open(f"{output_dir}/{filename}", 'w', encoding='utf-8') as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("\n----------Inference Complete!---------")

# --- ZIP FOR SUBMISSION ---
import shutil
shutil.make_archive("subtask_3", 'zip', output_dir)
print(f"Ready: subtask_3.zip")

Starting Inference Pipeline...

Processing eng-laptop...


  0%|          | 0/200 [00:00<?, ?it/s]

   Pair: perforemce + Great -> Score: 0.9997
   Pair: display + wide -> Score: 0.0041
   Pair: display + Very bright -> Score: 0.9996
   Pair: color gamut + wide -> Score: 0.9997
   Pair: color gamut + Very bright -> Score: 0.0012
   Pair: Battery life + bad -> Score: 0.9997
   Pair: Chromebook + very clean -> Score: 0.9997
   Pair: screen + clear -> Score: 0.9941
   Pair: screen + very bright -> Score: 0.9994
   Pair: laptop + excellent -> Score: 0.0610
   Pair: laptop + like -> Score: 0.9991
   Pair: sound quality + excellent -> Score: 0.9994
   Pair: sound quality + like -> Score: 0.0294
   Pair: Mid + very much worth -> Score: 0.9992
   Pair: video card + very much worth -> Score: 0.9996
   Pair: laptop + Great -> Score: 0.9996
   Pair: Zenbook + powerful -> Score: 0.9997
   Pair: Zenbook + [NULL] -> Score: 0.0006
   Pair: memory + powerful -> Score: 0.9715
   Pair: memory + [NULL] -> Score: 0.7944
   Pair: storage + powerful -> Score: 0.9906
   Pair: storage + [NULL] -> Score: 0.5

  0%|          | 0/200 [00:00<?, ?it/s]

   Pair: Food + great -> Score: 0.9996
   Pair: coffee + great -> Score: 0.9996
   Pair: Customer service + awesome -> Score: 0.0017
   Pair: Customer service + fantastic -> Score: 0.9997
   Pair: food + awesome -> Score: 0.9994
   Pair: food + fantastic -> Score: 0.0013
   Pair: Shrimp taco's + fresh -> Score: 0.0037
   Pair: Shrimp taco's + perfectly -> Score: 0.9991
   Pair: shrimp + fresh -> Score: 0.0049
   Pair: shrimp + perfectly -> Score: 0.9996
   Pair: Rolls + artfully made -> Score: 0.9996
   Pair: Rolls + reasonably priced -> Score: 0.9996
   Pair: Rolls + PACKED -> Score: 0.9095
   Pair: service + slow -> Score: 0.9996
   Pair: restaurant + tasty -> Score: 0.0026
   Pair: restaurant + good -> Score: 0.0040
   Pair: restaurant + pretty attentive -> Score: 0.0015
   Pair: restaurant + very clean -> Score: 0.9991
   Pair: bathrooms + tasty -> Score: 0.0016
   Pair: bathrooms + good -> Score: 0.0022
   Pair: bathrooms + pretty attentive -> Score: 0.0013
   Pair: bathrooms + ve